# Direct LLM Classifier — Three-Seed Test Evaluation

Clean public release for the MAPR 2026 paper **LLM-Augmented Hybrid Representations for Disease Category Classification from Clinical Notes**.

- Fixed split seed: `42`
- Experiment seeds: `42`, `123`, and `456`
- Dataset: credentialed MIMIC-IV-Ext-DiReCT v1.0.0
- Saved outputs are intentionally cleared from this release.


## Setup

This setup mirrors the rerun XGBoost notebook style and preserves Colab's preinstalled ML/data package versions. It installs only `rarfile`, matching the old notebooks. No system packages are installed.


In [ ]:
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Not in Colab; using local paths.')

if IN_COLAB:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', *['rarfile==4.2', 'openai==2.32.0', 'scikit-learn==1.6.1']],
        check=True,
    )


## Configuration

Default Colab paths:

- project root: `/content/drive/MyDrive/NCKH`
- data root: `/content/drive/MyDrive/NCKH/data/`
- cache root: `/content/drive/MyDrive/NCKH/cache/rerun_l1_3seeds/`
- results root: `/content/drive/MyDrive/NCKH/results/`


In [ ]:
import os, json, time, hashlib, concurrent.futures
from pathlib import Path
import rarfile
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
import openai

SPLIT_SEED = 42
RUN_SEEDS = [42, 123, 456]
OPENAI_MODEL = 'gpt-4o-mini'
OPENAI_TEMPERATURE = 0.0
EXTERNAL_LLM_DATA_COMPLIANCE_ACKNOWLEDGED = False
PROMPT_NAME = 'L1_DIRECT_REASONING'

if IN_COLAB:
    PROJECT_ROOT = Path(os.environ.get('LLM_FEATURES_PROJECT_ROOT', '/content/drive/MyDrive/LLM-features-clinical-notes'))
    EXTRACTED_ROOT = Path('/content/extracted_data')
else:
    PROJECT_ROOT = Path.cwd()
    EXTRACTED_ROOT = Path('/tmp/extracted_data')

DATA_ROOT = PROJECT_ROOT / 'data'
SPLIT_DIR = DATA_ROOT / 'splits'
CACHE_ROOT = PROJECT_ROOT / 'cache' / 'rerun_l1_3seeds'
RESULTS_ROOT = PROJECT_ROOT / 'results'
for p in [SPLIT_DIR, CACHE_ROOT, RESULTS_ROOT, EXTRACTED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('IN_COLAB:', IN_COLAB)
print('DATA_ROOT:', DATA_ROOT)
print('CACHE_ROOT:', CACHE_ROOT)
print('RESULTS_ROOT:', RESULTS_ROOT)


In [ ]:
def print_package_versions_for_audit():
    import importlib
    import platform

    package_modules = {
        'numpy': 'numpy',
        'pandas': 'pandas',
        'scikit-learn': 'sklearn',
        'openai': 'openai',
        'tqdm': 'tqdm',
        'rarfile': 'rarfile',
    }
    print('Package versions for audit:')
    print(f'  python: {platform.python_version()}')
    for package_name, module_name in package_modules.items():
        module = importlib.import_module(module_name)
        print(f"  {package_name}: {getattr(module, '__version__', 'unknown')}")


print_package_versions_for_audit()


## External LLM Access and Data Compliance

This notebook can send clinical-note text to an external LLM service. Before running the next cell, verify that your use complies with the current PhysioNet agreement and institutional requirements, including applicable zero-data-retention, no-training, and no-human-review protections.

After completing that review, set `EXTERNAL_LLM_DATA_COMPLIANCE_ACKNOWLEDGED = True` in the configuration cell. Store `OPENAI_API_KEY` in an environment variable or Colab Secrets; never paste it into the notebook.


In [ ]:
def require_external_llm_data_compliance():
    if not EXTERNAL_LLM_DATA_COMPLIANCE_ACKNOWLEDGED:
        raise RuntimeError(
            'Data-compliance acknowledgement is required before external LLM calls. '
            'Review the README and current PhysioNet guidance, then set '
            'EXTERNAL_LLM_DATA_COMPLIANCE_ACKNOWLEDGED = True in a separate cell.'
        )


require_external_llm_data_compliance()


def load_openai_api_key():
    key = os.environ.get('OPENAI_API_KEY')
    if key:
        return key
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get('OPENAI_API_KEY')
        except Exception:
            return None
    return None

OPENAI_API_KEY = load_openai_api_key()
if not OPENAI_API_KEY:
    raise RuntimeError('OPENAI_API_KEY is not set. In Colab, add it to Secrets as OPENAI_API_KEY.')
openai.api_key = OPENAI_API_KEY
openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)
print('OpenAI API key loaded.')


## Data Extraction and Exact Split Reconstruction

The clinical data are not distributed with this repository. Acquire MIMIC-IV-Ext-DiReCT v1.0.0 through PhysioNet and place `samples.rar` under `data/`.

The original experiment fixed `random_state=42`. Its input rows came from an unsorted Google Colab `os.walk`, whose traversal order is filesystem-dependent. The code below reconstructs that exact traversal using a 511-index permutation over a canonical lexicographic scan. This corrects a reproducibility issue; it does **not** select a favorable seed or alter the reported split.

The notebook validates the acquired file manifest, reconstructs the same two-stage stratified split, and checks the reported 308/101/102 sample counts and 25/23/23 category counts. Generated split CSVs remain local and are ignored by Git.


In [ ]:
def extract_samples_if_needed():
    rar_path = DATA_ROOT / 'samples.rar'
    if not rar_path.exists():
        raise FileNotFoundError(
            f'Missing {rar_path}. Acquire MIMIC-IV-Ext-DiReCT v1.0.0 from PhysioNet first.'
        )
    if (EXTRACTED_ROOT / 'Finished').exists() or (EXTRACTED_ROOT / 'samples').exists():
        print('Using existing extracted samples:', EXTRACTED_ROOT)
        return
    print('Extracting samples:', rar_path)
    with rarfile.RarFile(str(rar_path), 'r') as rf:
        rf.extractall(str(EXTRACTED_ROOT))


def find_finished_dir():
    for candidate in [EXTRACTED_ROOT / 'Finished', EXTRACTED_ROOT / 'samples']:
        if candidate.exists():
            return candidate
    for root, dirs, files in os.walk(EXTRACTED_ROOT):
        root_path = Path(root)
        if root_path.name in {'Finished', 'samples'}:
            return root_path
    raise FileNotFoundError(f'Cannot find Finished/ or samples/ under {EXTRACTED_ROOT}')


extract_samples_if_needed()
FINISHED_DIR = find_finished_dir()
print('FINISHED_DIR:', FINISHED_DIR)


# The original experiment fixed random_state=42 before evaluation. However,
# os.walk does not guarantee directory-entry order, so the same seed can yield
# different members when the input row order changes across filesystems.
#
# This 511-index permutation records the original Google Colab traversal order
# relative to a canonical lexicographic scan. It contains no clinical text,
# filenames, subject identifiers, labels, or precomputed split membership.
EXPECTED_SAMPLE_COUNT = 511
EXPECTED_CANONICAL_MANIFEST_SHA256 = 'd5173feb1a6ed1823c0723192af08862135b36349a261bc88747178575e7aa58'
COLAB_OS_WALK_ORDER_INDICES = [302, 301, 94, 91, 92, 87, 88, 85, 90, 86, 89, 93, 234, 239, 237, 216, 248, 236, 244, 242, 222,
 235, 224, 232, 231, 220, 221, 227, 246, 218, 229, 247, 211, 226, 208, 238, 213, 219, 217, 245,
 223, 243, 214, 212, 228, 240, 215, 230, 241, 209, 233, 210, 225, 65, 66, 67, 80, 77, 79, 76,
 75, 83, 84, 82, 78, 81, 71, 74, 72, 69, 68, 70, 73, 327, 318, 315, 319, 320, 317, 305, 312,
 304, 332, 330, 306, 309, 334, 307, 323, 322, 325, 308, 321, 314, 326, 331, 333, 311, 316, 324,
 310, 313, 329, 328, 303, 414, 408, 421, 409, 406, 407, 412, 410, 420, 413, 417, 411, 416, 415,
 418, 419, 425, 424, 422, 423, 480, 476, 487, 482, 483, 484, 481, 478, 477, 486, 479, 485, 488,
 462, 468, 463, 461, 467, 465, 464, 470, 474, 472, 471, 473, 469, 475, 466, 338, 337, 335, 336,
 265, 266, 293, 281, 251, 272, 249, 257, 261, 285, 283, 296, 256, 258, 262, 286, 300, 277, 252,
 254, 275, 270, 273, 282, 284, 290, 268, 288, 297, 289, 259, 263, 274, 278, 280, 253, 298, 291,
 271, 299, 255, 260, 264, 276, 294, 250, 295, 279, 292, 287, 267, 269, 109, 110, 111, 114, 120,
 121, 118, 119, 112, 113, 116, 115, 117, 447, 448, 459, 456, 458, 455, 454, 449, 457, 453, 452,
 460, 451, 450, 428, 444, 435, 431, 443, 438, 446, 437, 439, 427, 441, 429, 430, 426, 445, 433,
 436, 440, 442, 434, 432, 499, 500, 501, 503, 502, 339, 340, 341, 354, 357, 342, 353, 362, 359,
 347, 343, 349, 346, 351, 355, 360, 358, 361, 350, 345, 348, 356, 344, 352, 365, 364, 363, 134,
 148, 140, 137, 136, 139, 150, 149, 133, 138, 141, 135, 146, 145, 144, 132, 143, 147, 142, 169,
 167, 172, 171, 168, 170, 162, 164, 165, 160, 161, 163, 166, 510, 508, 509, 507, 506, 505, 504,
 386, 381, 393, 374, 378, 385, 384, 375, 392, 382, 388, 387, 379, 389, 377, 391, 380, 376, 390,
 383, 373, 370, 366, 369, 371, 368, 367, 372, 405, 397, 396, 403, 401, 404, 402, 400, 394, 395,
 399, 398, 496, 497, 498, 489, 490, 491, 492, 493, 495, 494, 130, 131, 127, 128, 124, 125, 129,
 122, 126, 123, 103, 104, 107, 101, 100, 105, 102, 106, 108, 96, 97, 99, 95, 98, 202, 195, 201,
 197, 200, 198, 196, 199, 206, 204, 205, 207, 203, 184, 181, 186, 191, 188, 182, 185, 192, 193,
 183, 190, 187, 194, 189, 176, 180, 177, 179, 178, 174, 173, 175, 59, 63, 55, 47, 44, 49, 51,
 62, 56, 61, 53, 50, 52, 64, 54, 57, 60, 43, 58, 46, 45, 48, 10, 9, 7, 3, 16, 22, 13, 2, 14, 23,
 17, 0, 21, 8, 19, 12, 4, 15, 1, 24, 18, 6, 26, 27, 20, 25, 11, 5, 35, 29, 42, 34, 38, 37, 36,
 41, 32, 33, 30, 39, 31, 40, 28, 156, 157, 151, 154, 153, 155, 152, 158, 159]


def _relative_path_from_file_path(file_path):
    normalized = str(file_path).replace('\\', '/')
    for marker in ['/Finished/', '/samples/']:
        if marker in normalized:
            return normalized.split(marker, 1)[1]
    return normalized


def collect_json_files_in_reported_order(finished_dir):
    canonical_rows = []
    finished_dir = Path(finished_dir)
    for root, dirs, files in os.walk(finished_dir):
        dirs.sort()
        for file_name in sorted(files):
            if not file_name.endswith('.json'):
                continue
            file_path = Path(root) / file_name
            relative_path = file_path.relative_to(finished_dir).as_posix()
            canonical_rows.append({
                'relative_path': relative_path,
                'file_path': str(file_path),
                'section': relative_path.split('/')[0],
            })
    canonical_rows.sort(key=lambda row: row['relative_path'])

    relative_paths = [row['relative_path'] for row in canonical_rows]
    if len(relative_paths) != EXPECTED_SAMPLE_COUNT:
        raise RuntimeError(
            f'Expected {EXPECTED_SAMPLE_COUNT} JSON files, found {len(relative_paths)}. '
            'Check that MIMIC-IV-Ext-DiReCT v1.0.0 was acquired and extracted correctly.'
        )
    if len(set(relative_paths)) != EXPECTED_SAMPLE_COUNT:
        raise RuntimeError('Duplicate relative paths found in the acquired dataset')

    manifest_hash = hashlib.sha256('\n'.join(relative_paths).encode('utf-8')).hexdigest()
    if manifest_hash != EXPECTED_CANONICAL_MANIFEST_SHA256:
        raise RuntimeError(
            'The acquired file manifest does not match the dataset version used in the paper. '
            f'Expected SHA-256 {EXPECTED_CANONICAL_MANIFEST_SHA256}, found {manifest_hash}.'
        )

    if len(COLAB_OS_WALK_ORDER_INDICES) != EXPECTED_SAMPLE_COUNT:
        raise RuntimeError('Recovered Colab order has the wrong length')
    if sorted(COLAB_OS_WALK_ORDER_INDICES) != list(range(EXPECTED_SAMPLE_COUNT)):
        raise RuntimeError('Recovered Colab order must contain each index from 0 to 510 exactly once')

    ordered_rows = [canonical_rows[index] for index in COLAB_OS_WALK_ORDER_INDICES]
    return pd.DataFrame([
        {'file_path': row['file_path'], 'section': row['section']}
        for row in ordered_rows
    ])


def create_fixed_split(seed=SPLIT_SEED):
    df_files = collect_json_files_in_reported_order(FINISHED_DIR)

    section_counts_initial = df_files['section'].value_counts()
    sections_for_train_only_initial = section_counts_initial[
        section_counts_initial < 3
    ].index.tolist()

    df_splittable_60_40 = df_files[
        ~df_files['section'].isin(sections_for_train_only_initial)
    ]
    df_non_stratifiable_initial = df_files[
        df_files['section'].isin(sections_for_train_only_initial)
    ]

    train_60, temp_40 = train_test_split(
        df_splittable_60_40,
        test_size=0.4,
        stratify=df_splittable_60_40['section'],
        random_state=seed,
    )

    section_counts_temp_40 = temp_40['section'].value_counts()
    sections_for_train_only_from_temp = section_counts_temp_40[
        section_counts_temp_40 < 2
    ].index.tolist()

    temp_40_splittable_50_50 = temp_40[
        ~temp_40['section'].isin(sections_for_train_only_from_temp)
    ]
    df_non_stratifiable_from_temp = temp_40[
        temp_40['section'].isin(sections_for_train_only_from_temp)
    ]

    val_20, test_20 = train_test_split(
        temp_40_splittable_50_50,
        test_size=0.5,
        stratify=temp_40_splittable_50_50['section'],
        random_state=seed,
    )

    train_df = pd.concat(
        [train_60, df_non_stratifiable_initial, df_non_stratifiable_from_temp],
        ignore_index=True,
    )
    assert len(train_df) + len(val_20) + len(test_20) == len(df_files)
    return (
        train_df.reset_index(drop=True),
        val_20.reset_index(drop=True),
        test_20.reset_index(drop=True),
    )


def _split_membership(df):
    return set(df['file_path'].apply(_relative_path_from_file_path))


def verify_loaded_splits_match_recreated(train_df, val_df, test_df):
    recreated = dict(zip(
        ['train', 'val', 'test'],
        create_fixed_split(SPLIT_SEED),
    ))
    loaded = {'train': train_df, 'val': val_df, 'test': test_df}
    for split_name in ['train', 'val', 'test']:
        loaded_membership = _split_membership(loaded[split_name])
        recreated_membership = _split_membership(recreated[split_name])
        if loaded_membership != recreated_membership:
            raise RuntimeError(
                f'Loaded {split_name}_split.csv does not match the reported split reconstruction.'
            )


def repair_path(path_value):
    raw = str(path_value)
    path = Path(raw)
    if path.exists():
        return str(path)
    parts = path.parts
    for marker in ['Finished', 'samples']:
        if marker in parts:
            candidate = FINISHED_DIR.joinpath(*parts[parts.index(marker) + 1:])
            if candidate.exists():
                return str(candidate)
    return raw


def load_or_create_splits():
    paths = {name: SPLIT_DIR / f'{name}_split.csv' for name in ['train', 'val', 'test']}
    if all(path.exists() for path in paths.values()):
        print('Loading local split CSVs from', SPLIT_DIR)
        train_df = pd.read_csv(paths['train'])
        val_df = pd.read_csv(paths['val'])
        test_df = pd.read_csv(paths['test'])
        verify_loaded_splits_match_recreated(train_df, val_df, test_df)
    else:
        print('Reconstructing the reported split with split seed', SPLIT_SEED)
        train_df, val_df, test_df = create_fixed_split(SPLIT_SEED)
        train_df.to_csv(paths['train'], index=False)
        val_df.to_csv(paths['val'], index=False)
        test_df.to_csv(paths['test'], index=False)
    for df in [train_df, val_df, test_df]:
        df['file_path'] = df['file_path'].apply(repair_path)
    return train_df, val_df, test_df


train_df, val_df, test_df = load_or_create_splits()
split_sizes = (len(train_df), len(val_df), len(test_df))
split_category_counts = tuple(
    df['section'].nunique() for df in [train_df, val_df, test_df]
)
assert split_sizes == (308, 101, 102), split_sizes
assert split_category_counts == (25, 23, 23), split_category_counts
print('Split sizes:', split_sizes)
print('Category counts:', split_category_counts)


## Clinical Notes And Disease Options

L1 is a direct reasoning baseline. It does not use train examples for model fitting, but the train split is loaded so `LabelEncoder` and the disease-option list are consistent with the main XGBoost experiments.


In [ ]:
def extract_clinical_note(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    clinical_note_parts = []
    for i in range(1, 7):
        input_key = f'input{i}'
        value = data.get(input_key)
        if value:
            text = str(value).strip()
            if text and text.lower() != 'n/a':
                clinical_note_parts.append(text)
    return '\n'.join(clinical_note_parts)


def load_notes_and_labels(df):
    notes = []
    labels = []
    file_paths = []
    missing = []
    for _, row in df.iterrows():
        file_path = Path(row['file_path'])
        if not file_path.exists():
            missing.append(str(file_path))
            continue
        notes.append(extract_clinical_note(file_path))
        labels.append(row['section'])
        file_paths.append(str(file_path))
    if missing:
        raise FileNotFoundError(f'{len(missing)} missing files. First: {missing[0]}')
    return notes, labels, file_paths


# Train labels are needed for consistent label encoding and disease options.
X_train_text, y_train, train_paths = load_notes_and_labels(train_df)
X_test_text, y_test, test_paths = load_notes_and_labels(test_df)

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

disease_options = [str(name) for name in label_encoder.classes_]
disease_option_to_id = {name: idx for idx, name in enumerate(disease_options)}
all_labels = np.arange(len(disease_options))
all_target_names = disease_options

assert len(X_test_text) == len(y_test_encoded) == len(test_paths)
print('Test examples:', len(X_test_text))
print('Disease options:')
for option in disease_options:
    print('  -', option)


## L1 Prompt And Cache Helpers

The prompt template follows the D* settings in the DiReCT paper.

In [ ]:
SYSTEM_PROMPT = 'Suppose you are one of the greatest AI scientists and medical expert. Let us think step by step.'
USER_PROMPT_TEMPLATE = """You will review a clinical 'Note' and your 'Response' is to diagnose the disease that the patient have for this admission.
All possible disease options are in a list structure: {disease_options}.

Note that you can only choose one disease from the disease options and directly output the origin name of that disease.
Now, start to complete your task.
Don't output any information other than your 'Response'.

'Note':
{clinical_note}

Your 'Response':"""


def stable_hash(value):
    payload = json.dumps(value, ensure_ascii=False, separators=(',', ':'))
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


def prompt_hash():
    return stable_hash({'system': SYSTEM_PROMPT, 'user': USER_PROMPT_TEMPLATE})


def prediction_cache_path(split_name, seed):
    return CACHE_ROOT / f'l1_predictions_{OPENAI_MODEL}_seed{seed}_{split_name}.json'


def expected_cache_metadata(texts, file_paths, split_name, seed):
    return {
        'model': OPENAI_MODEL,
        'split_name': split_name,
        'seed': seed,
        'temperature': OPENAI_TEMPERATURE,
        'prompt_hash': prompt_hash(),
        'disease_options_hash': stable_hash(disease_options),
        'texts_hash': stable_hash(texts),
        'file_path_hash': stable_hash(file_paths),
    }


def validate_prediction_cache(payload, texts, file_paths, split_name, seed):
    expected = expected_cache_metadata(texts, file_paths, split_name, seed)
    mismatches = {
        key: {'expected': value, 'found': payload.get(key)}
        for key, value in expected.items()
        if payload.get(key) != value
    }
    if mismatches:
        raise ValueError(
            f'L1 cache metadata mismatch for {split_name} seed {seed}: {json.dumps(mismatches, ensure_ascii=False, indent=2)}'
        )
    predictions = payload.get('predictions')
    if not isinstance(predictions, list) or len(predictions) != len(texts):
        raise ValueError(
            f'L1 cache prediction count mismatch for {split_name} seed {seed}: expected {len(texts)}, found {0 if predictions is None else len(predictions)}'
        )


## OpenAI Prediction Helpers

The helper mirrors the old notebook's retry style and uses parallel calls for speed. Results are stored back by original row index, so cached predictions preserve split order.


In [ ]:
def _get_single_l1_prediction_openai(clinical_note, disease_options, seed, max_attempts=3, delay_between_attempts=5):
    require_external_llm_data_compliance()
    generated_prediction = 'Error: OpenAI API call failed'
    prompt_tokens_count = 0
    completion_tokens_count = 0
    for attempt in range(max_attempts):
        try:
            user_prompt = USER_PROMPT_TEMPLATE.format(
                disease_options=disease_options,
                clinical_note=clinical_note,
            )
            response = openai_client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': user_prompt},
                ],
                temperature=OPENAI_TEMPERATURE,
                seed=seed,
            )
            if response.choices and response.choices[0].message.content:
                generated_prediction = response.choices[0].message.content.strip()
                if response.usage:
                    prompt_tokens_count = response.usage.prompt_tokens
                    completion_tokens_count = response.usage.completion_tokens
                return generated_prediction, prompt_tokens_count, completion_tokens_count
            print(f'Warning: Empty or malformed response, attempt {attempt + 1}. Retrying...')
            time.sleep(delay_between_attempts)
        except openai.APITimeoutError:
            print(f'Timeout error, attempt {attempt + 1}. Retrying in {delay_between_attempts}s...')
            time.sleep(delay_between_attempts)
        except openai.APIConnectionError as exc:
            print(f'Connection error, attempt {attempt + 1}: {exc}. Retrying in {delay_between_attempts}s...')
            time.sleep(delay_between_attempts)
        except openai.RateLimitError as exc:
            if 'insufficient_quota' in str(exc):
                print(f'Quota error: {exc}. Please check your OpenAI plan and rerun. Skipping retries for this sample.')
                return 'Error: Insufficient Quota', 0, 0
            print(f'Rate limit hit, attempt {attempt + 1}. Waiting longer ({delay_between_attempts * 2}s) before retrying...')
            time.sleep(delay_between_attempts * 2)
        except Exception as exc:
            print(f'Error generating L1 prediction, attempt {attempt + 1}: {exc}. Retrying in {delay_between_attempts}s...')
            time.sleep(delay_between_attempts)
    return generated_prediction, prompt_tokens_count, completion_tokens_count


def get_l1_predictions_openai(texts, file_paths, split_name, seed, max_workers=3, max_attempts=3, delay_between_attempts=5, force_refresh=False):
    cache_path = prediction_cache_path(split_name, seed)
    if cache_path.exists() and not force_refresh:
        payload = json.loads(cache_path.read_text(encoding='utf-8'))
        validate_prediction_cache(payload, texts, file_paths, split_name, seed)
        print(f'Loaded L1 predictions for {split_name} seed {seed}:', cache_path)
        return payload['predictions']

    predictions = [None] * len(texts)
    items = [None] * len(texts)
    total_prompt_tokens = 0
    total_completion_tokens = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_index = {
            executor.submit(
                _get_single_l1_prediction_openai,
                text,
                disease_options,
                seed,
                max_attempts,
                delay_between_attempts,
            ): i
            for i, text in enumerate(texts)
        }
        for future in tqdm(concurrent.futures.as_completed(future_to_index), total=len(texts), desc=f'L1 {split_name} seed {seed}'):
            index = future_to_index[future]
            try:
                prediction, prompt_tokens, completion_tokens = future.result()
                predictions[index] = prediction
                total_prompt_tokens += prompt_tokens
                total_completion_tokens += completion_tokens
                items[index] = {
                    'index': index,
                    'file_path': file_paths[index],
                    'prediction': prediction,
                    'prompt_tokens': prompt_tokens,
                    'completion_tokens': completion_tokens,
                }
            except Exception as exc:
                prediction = 'Error: Parallel processing exception'
                predictions[index] = prediction
                items[index] = {'index': index, 'file_path': file_paths[index], 'prediction': prediction, 'error': repr(exc)}

    payload = expected_cache_metadata(texts, file_paths, split_name, seed)
    payload.update({
        'predictions': predictions,
        'items': items,
        'prompt_tokens': total_prompt_tokens,
        'completion_tokens': total_completion_tokens,
    })
    cache_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Saved L1 predictions for {split_name} seed {seed}:', cache_path)
    return predictions


## Evaluation Helpers

Predictions are stripped before scoring. If a model output is not exactly one of the disease-option labels, it is recorded as invalid and counted as incorrect.


In [ ]:
def clean_prediction(value):
    return str(value).strip()


def predictions_to_encoded(predictions):
    encoded = []
    valid = []
    cleaned = []
    for value in predictions:
        pred = clean_prediction(value)
        cleaned.append(pred)
        if pred in disease_option_to_id:
            encoded.append(disease_option_to_id[pred])
            valid.append(True)
        else:
            encoded.append(-1)
            valid.append(False)
    return np.array(encoded), cleaned, valid


def evaluate_l1_split(split_name, seed, texts, labels, y_true_encoded, file_paths):
    predictions = get_l1_predictions_openai(texts, file_paths, split_name, seed)
    y_pred_encoded, cleaned_predictions, valid_flags = predictions_to_encoded(predictions)

    row = {
        'method': 'L1 reasoning baseline',
        'split': split_name,
        'run_seed': seed,
        'llm_seed': seed,
        'model': OPENAI_MODEL,
        'temperature': OPENAI_TEMPERATURE,
        'n_examples': len(labels),
        'n_valid_predictions': int(sum(valid_flags)),
        'n_invalid_predictions': int(len(valid_flags) - sum(valid_flags)),
        'accuracy': accuracy_score(y_true_encoded, y_pred_encoded),
        # Match the XGBoost rerun headline metrics: sklearn infers labels from y_true/y_pred.
        'f1_micro': f1_score(y_true_encoded, y_pred_encoded, average='micro', zero_division=0),
        'f1_macro': f1_score(y_true_encoded, y_pred_encoded, average='macro', zero_division=0),
    }
    report = classification_report(
        y_true_encoded,
        y_pred_encoded,
        labels=all_labels,
        target_names=all_target_names,
        output_dict=True,
        zero_division=0,
    )

    prediction_rows = []
    for file_path, actual, raw_pred, clean_pred, is_valid in zip(file_paths, labels, predictions, cleaned_predictions, valid_flags):
        prediction_rows.append({
            'split': split_name,
            'run_seed': seed,
            'file_path': file_path,
            'actual': actual,
            'predicted_raw': raw_pred,
            'predicted_clean': clean_pred,
            'is_valid_label': is_valid,
            'correct': clean_pred == actual,
        })
    return row, report, prediction_rows


def summarize_l1_results(metrics_df):
    metrics = ['accuracy', 'f1_micro', 'f1_macro']
    rows = []
    for split_name, group in metrics_df.groupby('split', sort=False):
        row = {'split': split_name, 'n_runs': len(group)}
        for metric in metrics:
            row[f'{metric}_mean'] = group[metric].mean()
            row[f'{metric}_std'] = group[metric].std(ddof=1)
        rows.append(row)
    return pd.DataFrame(rows)


## Run L1 Evaluation

This loop evaluates the fixed test split for all configured seeds.


In [ ]:
all_metric_rows = []
all_prediction_rows = []
all_reports = {}

eval_split = ('test', X_test_text, y_test, y_test_encoded, test_paths)

for run_seed in RUN_SEEDS:
    print(f'\n=== L1 RUN SEED {run_seed} ===')
    split_name, texts, labels, y_true_encoded, file_paths = eval_split
    row, report, prediction_rows = evaluate_l1_split(split_name, run_seed, texts, labels, y_true_encoded, file_paths)
    all_metric_rows.append(row)
    all_prediction_rows.extend(prediction_rows)
    all_reports[f'{split_name}__seed{run_seed}'] = report
    print(
        f"{split_name}: accuracy={row['accuracy']:.4f}, "
        f"f1_micro={row['f1_micro']:.4f}, f1_macro={row['f1_macro']:.4f}, "
        f"invalid={row['n_invalid_predictions']}"
    )

metrics_df = pd.DataFrame(all_metric_rows)
predictions_df = pd.DataFrame(all_prediction_rows)
assert len(metrics_df) == len(RUN_SEEDS), f'Expected {len(RUN_SEEDS)} metric rows, got {len(metrics_df)}'
metrics_df


## Save Results

This writes outputs to `RESULTS_ROOT`, which is usually `/content/drive/MyDrive/NCKH/results/` in Colab.

Files written:

- `rerun_l1_reasoning_3seeds_test_xgbmetrics_predictions.csv`
- `rerun_l1_reasoning_3seeds_test_xgbmetrics_detailed.csv`
- `rerun_l1_reasoning_3seeds_test_xgbmetrics_summary.csv`
- `rerun_l1_reasoning_3seeds_test_xgbmetrics_results.xlsx`, if `openpyxl` is already available
- `rerun_l1_reasoning_3seeds_test_xgbmetrics_classification_reports.json`


In [ ]:
predictions_path = RESULTS_ROOT / 'rerun_l1_reasoning_3seeds_test_xgbmetrics_predictions.csv'
detailed_path = RESULTS_ROOT / 'rerun_l1_reasoning_3seeds_test_xgbmetrics_detailed.csv'
summary_path = RESULTS_ROOT / 'rerun_l1_reasoning_3seeds_test_xgbmetrics_summary.csv'
excel_path = RESULTS_ROOT / 'rerun_l1_reasoning_3seeds_test_xgbmetrics_results.xlsx'
reports_path = RESULTS_ROOT / 'rerun_l1_reasoning_3seeds_test_xgbmetrics_classification_reports.json'

predictions_df.to_csv(predictions_path, index=False)
metrics_df.to_csv(detailed_path, index=False)
summary_df = summarize_l1_results(metrics_df)
summary_df.to_csv(summary_path, index=False)
reports_path.write_text(json.dumps(all_reports, ensure_ascii=False, indent=2), encoding='utf-8')

try:
    import openpyxl  # noqa: F401
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        predictions_df.to_excel(writer, sheet_name='predictions', index=False)
        metrics_df.to_excel(writer, sheet_name='detailed', index=False)
        summary_df.to_excel(writer, sheet_name='summary', index=False)
    print('Saved Excel:', excel_path)
except Exception as exc:
    print('Excel export skipped:', exc)

print('Saved predictions CSV:', predictions_path)
print('Saved detailed metrics CSV:', detailed_path)
print('Saved summary CSV:', summary_path)
print('Saved classification reports JSON:', reports_path)
display(summary_df)


## Final Sanity Checks

The notebook should produce exactly three detailed metric rows and one summary row:

- `3` detailed rows: one for each seed `42`, `123`, and `456`.
- `1` summary row.

If any assertion fails, do not use the result files until the cause is understood.


In [ ]:
print('Detailed metric rows:', len(metrics_df))
print('Summary rows:', len(summary_df))
print('Splits:', sorted(metrics_df['split'].unique()))
print('Seeds:', sorted(metrics_df['run_seed'].unique()))
assert len(metrics_df) == len(RUN_SEEDS)
assert len(summary_df) == 1
assert sorted(metrics_df['split'].unique()) == ['test']
assert sorted(metrics_df['run_seed'].unique()) == RUN_SEEDS
metric_cols = ['accuracy', 'f1_micro', 'f1_macro']
assert np.isfinite(metrics_df[metric_cols].to_numpy()).all()
assert ((metrics_df[metric_cols] >= 0) & (metrics_df[metric_cols] <= 1)).all().all()
assert (summary_df['n_runs'] == len(RUN_SEEDS)).all()
assert 'train' not in set(metrics_df['split'])
print('Final checks passed.')
